In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


In [ ]:
# Reading the saved data pickle file
# rng = pd.date_range('2019-02-24', periods=661, freq='T')
# df  = pd.DataFrame()
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentiment.pkl', compression=None)
df_companies=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/press_release_companies_statistics.pkl', compression=None)

In [ ]:
df_companies

In [ ]:
df["articles"].describe()
df = df.join(df_sentiment)
df["vix"] = df["adj close"]#.apply(np.int)
df["vix_ln_return"]=np.log(df["vix"]/df["vix"].shift(1))
df['sentiment'] = df[["sentiment"]].apply(pd.to_numeric)
df["sentiment_ewm"]=df["sentiment"].ewm(10).mean()
df["date"]=df.index
# df['date'] = pd.date_range(start='25/9/2019', periods=len(df), freq='D')
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-07-15']

In [ ]:
df

In [ ]:
df_companies.plot(kind="hist")

In [ ]:
df_companies

In [ ]:
df_companies = df_companies.rename(columns={"companies": "Ticker", "count": "PressReleaseCount"})

In [ ]:
df_companies

In [ ]:
# cols = (df_companies[["companies", "count"]]
#         .groupby("companies")
#         .mean()
#         .sort_values(by="count", ascending=False)[:150].index)
# best_features = df_companies.loc[feature_importance_df.Feature.isin(cols)]

plt.figure(figsize=(20,150))
chart=sns.barplot(x="PressReleaseCount", y="Ticker", data=df_companies.sort_values(by="PressReleaseCount",ascending=False))
chart.bar_label(chart.containers[0])
plt.title('S&P500 Companies by Press Release Count')
plt.tight_layout()
plt.savefig('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/S&P500 Companies by Press Release Count.png')

In [ ]:
df_companies_mkp=pd.read_csv('/Users/Farid Srouji/Documents/Research/master/globaldata/S&P500CompaniesByMarketCap.csv')

In [ ]:
df_companies_mkp

In [ ]:
type(df_companies_mkp["MarketCapMillions"])

In [ ]:
df_companies_mkp = df_companies_mkp#.loc[1:3]#].Company.array
df_companies_mkp=df_companies_mkp[["Ticker", "Industry", "MarketCapMillions"]]

In [ ]:
type(pd.to_numeric(df_companies_mkp['MarketCapMillions']))

In [ ]:
df_companies_mkp

In [ ]:
import plotly.express as px
import numpy as np
fig = px.treemap(df_companies_mkp, path=[px.Constant("MarketCapMillions"), 'Industry', 'Ticker'], values='MarketCapMillions',
                  color='MarketCapMillions', hover_data=['MarketCapMillions'],
                  color_continuous_scale='RdBu')
                  #color_continuous_midpoint=np.average(df_companies_mkp['MarketCapMillions'], weights=df_companies_mkp['MarketCapMillions']))
fig.update_layout(margin = dict(t=100, l=50, r=50, b=50))
fig.show()

In [ ]:
df_companies

In [ ]:
import plotly.express as px
import numpy as np
fig = px.treemap(df_companies, path=[px.Constant("PressReleaseCount"), 'Ticker'], values='PressReleaseCount',
                  color='PressReleaseCount', hover_data=['PressReleaseCount'],
                  color_continuous_scale='RdBu')
                  #color_continuous_midpoint=np.average(df_companies_mkp['MarketCapMillions'], weights=df_companies_mkp['MarketCapMillions']))
fig.update_layout(margin = dict(t=100, l=50, r=50, b=50))
fig.show()

In [ ]:
import plotly.express as px
import numpy as np
df = px.data.gapminder().query("year == 2007")
print(df)
fig = px.treemap(df, path=[px.Constant("world"), 'continent', 'country'], values='pop',
                  color='lifeExp', hover_data=['iso_alpha'],
                  color_continuous_scale='RdBu',
                  color_continuous_midpoint=np.average(df['lifeExp'], weights=df['pop']))
fig.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig.show()